# What krovlab cannot do (yet), and where it goes wrong

The roofs this library generates are a **strict subset** of roofs you
can build. Two different kinds of "no" show up:

1. **The method cannot represent it.** A chimney through the slope, a
   true circular wall, a ridge layout that is not "these walls are one
   plane". Knee height, gambrel, wrap, and dormers are knobs — see
   `getting-started.ipynb`. Without a wrap, the library still puts a
   hip or a valley at each corner.
2. **The implementation returns a `Roof` that fails the terrain check.**
   Input was accepted, a skeleton was produced, and `validity.is_terrain`
   is false. The quantities are unusable. This is not an exception and
   not always a `Failure`. A **dormer** project is also not a terrain
   (the host has a hole), but that is the documented exception:
   quantities still add up and 3D still draws. A plus-shape still hides
   3D.

Drawings in this notebook:

- A **valid** roof gets a plan (outline + skeleton) and a 3D solid.
- A **constructed but invalid** roof gets the plan of what came out —
  so you can see the holes in the covering. No 3D: the solid would
  pretend the missing faces are not missing.
- A **dormer project** gets plan and 3D even though it is not a terrain.
- A **`Failure`** gets the input outline in red. There is no roof.

Always read `validity` before trusting a result. The longer write-up is
[`docs/limitations.md`](../docs/limitations.md).


In [ ]:
from collections import defaultdict

import plotly.graph_objects as go

from krovlab import Cell, Dormer, Failure, Project, Roof, project, roof
from krovlab.viz import plan_view, solid_view


def describe(result: Roof | Project | Failure) -> None:
    if isinstance(result, Failure):
        print(f"Failure  kind={result.kind}\n  {result.reason}")
        return
    by_kind: dict[str, int] = defaultdict(int)
    for arc in result.arcs:
        by_kind[arc.kind] += 1
    print(
        f"Roof  terrain={result.validity.is_terrain}  "
        f"h={result.ridge_height:.3f} m  "
        f"faces={len(result.faces)}  arcs={dict(by_kind)}"
    )
    if result.validity.is_terrain:
        print(f"  sloped area {result.total_sloped_area:.3f} m²")
        return
    seen: set[str] = set()
    for reason in result.validity.reasons:
        head = reason.split(":")[0]
        if head in seen:
            continue
        seen.add(head)
        print(f"  !! {reason}")


def footprint_outline(
    footprint: list[tuple[float, float]],
    holes: list[list[tuple[float, float]]] | None = None,
    title: str = "",
) -> go.Figure:
    fig = go.Figure()
    if footprint:
        xs = [p[0] for p in footprint] + [footprint[0][0]]
        ys = [p[1] for p in footprint] + [footprint[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines+markers",
                name="footprint",
                line={"color": "#b91c1c", "width": 2},
                fill="toself",
                fillcolor="rgba(185, 28, 28, 0.10)",
                marker={"size": 7, "color": "#b91c1c"},
            )
        )
    for i, hole in enumerate(holes or []):
        if not hole:
            continue
        xs = [p[0] for p in hole] + [hole[0][0]]
        ys = [p[1] for p in hole] + [hole[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines+markers",
                name=f"hole {i}" if len(holes or []) > 1 else "hole",
                line={"color": "#1d4ed8", "width": 2, "dash": "dot"},
                marker={"size": 6, "color": "#1d4ed8"},
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="x (m)",
        yaxis_title="y (m)",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_white",
        height=420,
        legend_title="input",
    )
    return fig


def show(
    title: str,
    result: Roof | Project | Failure,
    footprint: list[tuple[float, float]] | None = None,
    holes: list[list[tuple[float, float]]] | None = None,
) -> None:
    """Valid roof: plan + 3D. Dormer project: plan + 3D. Other non-terrain:
    plan of what came out. Failure: input outline in red."""
    print(f"=== {title} ===")
    describe(result)
    print()
    dormer_solid = (
        isinstance(result, Project)
        and not result.validity.is_terrain
        and all("dormer" in reason.lower() for reason in result.validity.reasons)
    )
    if isinstance(result, (Roof, Project)) and (
        result.validity.is_terrain or dormer_solid
    ):
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (brown = walls, eave = roof edge)")
        plan.show()
        solid = solid_view(result, walls=footprint, wall_holes=holes)
        caption = (
            f"{title} — 3D (dormer: not a terrain, solid still draws)"
            if dormer_solid
            else f"{title} — 3D (brown = walls at height 0)"
        )
        solid.update_layout(title=caption)
        solid.show()
    elif isinstance(result, (Roof, Project)):
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (not a terrain)")
        plan.show()
    elif footprint is not None:
        footprint_outline(
            footprint,
            holes,
            title=f"{title} — {result.kind}",
        ).show()


## Always check `validity.is_terrain`

A `Failure` means no roof was produced. A `Roof` with
`is_terrain == False` means a roof *was* produced and then failed the
checks. Both are "do not use"; only the second looks like success if
you only check `isinstance(..., Roof)`. A **dormer** project is the
exception: not a terrain, but covering numbers still add up and 3D
still draws.

Simple convex, L and U footprints at one pitch typically pass. The
cells below are the ones that currently do not — plus the dormer
contrast after the plus-shape.


## Crossing arms (a plus-shaped plan)

Several split events collide at once. The call returns a `Roof`, never
an exception, but the faces do not cover the footprint. Four of the
twelve faces are just an eave with no area.


In [ ]:
plus = [
    (2.0, 0.0), (4.0, 0.0), (4.0, 2.0), (6.0, 2.0),
    (6.0, 4.0), (4.0, 4.0), (4.0, 6.0), (2.0, 6.0),
    (2.0, 4.0), (0.0, 4.0), (0.0, 2.0), (2.0, 2.0),
]
plus_roof = roof(plus, 45.0)
show("plus at 45°", plus_roof, plus)
if isinstance(plus_roof, Roof):
    for face in plus_roof.faces:
        print(f"  edge {face.edge_index}  {len(face.node_indices)} verts  plan {face.plan_area:.3f} m²")


## A dormer is not a terrain — 3D still draws

A plus-shape hides 3D because the covering is broken. A dormer is a
deliberate hole: `validity.is_terrain` is false, the describe block says
so, and the solid still builds (host with an opening, dormer on top).
That is the documented exception, not a quiet weakening of the
plus-shape case.


In [ ]:
rect = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
dormer_ring = [(4.0, 0.5), (6.0, 0.5), (6.0, 2.0), (4.0, 2.0)]
dormered = project(
    [Cell(rect, 45.0)],
    [Dormer(0, dormer_ring, [45.0, 90.0, 45.0, 90.0])],
)
show("gable dormer on the south slope (not a terrain, 3D still draws)", dormered, rect)
print("plus-shape above: plan only. dormer: plan and 3D.")


## T-shapes are pitch-sensitive

The same T-plan is a valid terrain at 45° and not at 30°. The wavefront
events all fire at one instant; at some pitches the ridges between
those nodes are never drawn, and a chunk of the plan is left unroofed.

Do not assume that a T which worked at one pitch will work at another.


In [ ]:
t_shape = [
    (0.0, 6.0), (4.0, 6.0), (4.0, 0.0), (8.0, 0.0),
    (8.0, 6.0), (12.0, 6.0), (12.0, 10.0), (0.0, 10.0),
]
for pitch in (30.0, 45.0, 60.0):
    show(f"T at {pitch:g}°", roof(t_shape, pitch), t_shape)


A slightly different T — a 12 × 6 m bar with a 4 × 4 m stem — currently
fails at every pitch tried here, including 45°.


In [ ]:
house = [
    (0.0, 0.0), (12.0, 0.0), (12.0, 6.0), (8.0, 6.0),
    (8.0, 10.0), (4.0, 10.0), (4.0, 6.0), (0.0, 6.0),
]
for pitch in (20.0, 45.0, 60.0):
    show(f"house-T at {pitch:g}°", roof(house, pitch), house)


## Extra vertex on a straight wall

The method gives **one face per footprint edge**. A midpoint on an
otherwise straight eave is two edges, so you get two faces of the
same plane — never one plane spanning both. Same pitch on both
halves is a valid terrain; the extra vertex traces inland
perpendicular to the wall. Leave a straight wall as two endpoints
unless you want that split.


In [ ]:
collinear = [(0.0, 0.0), (5.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
clean_rect = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
show("collinear extra vertex", roof(collinear, 45.0), collinear)
show("same rectangle, no extra point", roof(clean_rect, 45.0), clean_rect)


## One straight wall cannot carry two pitches

Adjacent collinear edges of **differing** pitch are not a terrain:
both planes contain the wall line at eave height and conflict inland.
There is no unique weighted skeleton (Biedl et al., 2015); the
library refuses up front as `unsupported`.


In [ ]:
show(
    "collinear extra vertex, two pitches",
    roof(collinear, [45.0, 30.0, 45.0, 45.0, 45.0]),
    collinear,
)


## Mixed pitch on some L-shapes

A different pitch per edge is the feature the project is named for, and
it works on convex footprints and on some L-shapes. A large gap between
a shallow face and a steep neighbour on an L can still come back as a
non-terrain `Roof` — not as `Failure(incomplete)`.


In [ ]:
l_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 6.0),
    (3.0, 6.0), (3.0, 10.0), (0.0, 10.0),
]
show("L, uniform 45°", roof(l_shape, 45.0), l_shape)
show("L, mild mix", roof(l_shape, [45.0, 30.0, 45.0, 60.0, 45.0, 30.0]), l_shape)
show("L, steep/shallow neighbours", roof(l_shape, [45.0, 15.0, 45.0, 75.0, 45.0, 45.0]), l_shape)


## A gable on some edges of an L or a U

Gabling a short side of a **rectangle** is solid. Gabling some edges of
an L or a U currently fails the arc-classification check: a verge is
drawn that does not lie on the gable wall. Other edges of the same
footprint gable cleanly. Try the edge you care about, and read
`validity`.


In [ ]:
rectangle = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
show("rectangle, east gable", roof(rectangle, [45.0, 90.0, 45.0, 45.0]), rectangle)

for i in range(6):
    pitches = [45.0] * 6
    pitches[i] = 90.0
    show(f"L gable on edge {i}", roof(l_shape, pitches), l_shape)


## Input that is refused (these *are* `Failure`)

The cases below never produce a `Roof`. Each is drawn as the input
outline. A caller processing many footprints can branch on `kind` and
keep going.


In [ ]:
square = [(0.0, 0.0), (10.0, 0.0), (10.0, 10.0), (0.0, 10.0)]
bowtie = [(0.0, 0.0), (10.0, 10.0), (10.0, 0.0), (0.0, 10.0)]
line = [(0.0, 0.0), (10.0, 0.0), (4.0, 0.0)]
touching_hole = [(0.0, 0.0), (4.0, 0.0), (4.0, 4.0), (0.0, 4.0)]
u_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 8.0), (7.0, 8.0),
    (7.0, 3.0), (3.0, 3.0), (3.0, 8.0), (0.0, 8.0),
]

show("pitch 0", roof(square, 0.0), square)
show("every edge a gable", roof(square, 90.0), square)
show("bowtie", roof(bowtie, 45.0), bowtie)
show("a line", roof(line, 45.0), line)
show("hole touching the outer ring", roof(square, 45.0, holes=[touching_hole]), square, [touching_hole])
show("overhang that folds a U", roof(u_shape, 45.0, overhang=2.5), u_shape)


## Inherent to the method

These will not go away by fixing bugs. They are why a passing roof is
still only one of the roofs that could stand on those walls.

### One face per footprint edge — unless you wrap

A single plane that continues across two or more consecutive walls is
**wrap**: mark those edges as one plane. Without a wrap, the library
puts a hip or a valley at each corner. Alternate ridge layouts that
are not wrap stay out; use a gable mask or a project of several cells.

### One straight wall, one pitch

Two different pitches on collinear eaves of the same wall are not a
terrain, and the weighted skeleton has no unique answer. The library
refuses that input. Same-pitch extra vertices are two coplanar faces,
not one spanning plane. A gambrel is two pitches **up the slope**, not
along the facade — that is a different knob.

### One style among several

The same walls admit more than one valid roof. This library produces
the fully hipped roof at the pitches you gave, or a mixed hip-and-gable
roof when you set an edge to 90°. A Dutch gable and a barn break are
knee height and gambrel on an edge. A different ridge layout on the
same plan is not generated.

### Not in the model

Mansards as a four-wall break, butterfly roofs, chimneys, rooflights,
curved walls, and split-level eaves on one cell are not modelled.
Dormers *are* modelled: a small ring on a host face, not a cell at eave
height. Two eave heights are two cells in a `project`. A pitched shared
wall at two heights, or a gable against a pitch, is a named Failure.
Disconnected wings are also a `project` of two cells, not one skeleton
over the outer wall.

See [`docs/limitations.md`](../docs/limitations.md) and
[`docs/future-work.md`](../docs/future-work.md).


## Circles and chimneys

A hole must be a polygon. A circle is an n-gon approximation — see
`getting-started.ipynb`. Each segment is still one wall.

A hole is a courtyard at **eave height**, with inward faces unless you
gable those edges. A **chimney** would cut the roof *above* the eaves,
through the slope. That is a penetration, and it is not in the model —
a **dormer** is the penetration that is. Gabling the hole (`pitch = 90`
on every inner edge) gives vertical courtyard walls, still opening at
the eave plane — a light well, not a chimney.


## What a drawing can hide

`plan_view` fills the **outer** ring. On a courtyard the hole's eaves,
valleys and ridges are drawn, but the gray fill covers the courtyard
too — it is not punched out. Read the eave and valley traces, or open
the 3D solid, rather than trusting the fill.


In [ ]:
courtyard = [(3.0, 3.0), (7.0, 3.0), (7.0, 7.0), (3.0, 7.0)]
show("courtyard (fill does not punch the hole)", roof(square, 45.0, holes=[courtyard]), square, [courtyard])


The plus-shaped plan above is the other drawing trap: `plan_view` will
happily draw a skeleton that does not cover the building. The picture
is not a substitute for `validity.is_terrain`.


## What to trust first

The tests generate convex polygons, L-shapes, U-shapes, and rectangles
with rectangular holes, at uniform pitch, with modest overhang, and
(on convex footprints) per-edge pitch and a gable or two. Worked
examples lock the square, the 10 × 6 m rectangle, the L with one
valley, the centred courtyard, and a gabled rectangle.

Those are the shapes to start from, plus the worked knee, gambrel,
wrap, and dormer examples in `getting-started.ipynb`. For anything
else: call `roof` or `project`, then require a result that is not a
`Failure`. If it is a `Roof` or `Project`, read `validity.is_terrain`
before you keep the quantities — a dormer project is the one case
where a false terrain flag still has usable covering and a 3D solid.
